In [1]:
import os
import glob
import torch
from PIL import Image, ImageDraw, ImageFont
import numpy as np

class ColorMapper:
    def __init__(self):
        self.colors = {}

    def get_color(self, class_name):
        if class_name not in self.colors:
            h = hash(class_name)
            r = (h & 0xFF0000) >> 16
            g = (h & 0x00FF00) >> 8
            b = (h & 0x0000FF)
            self.colors[class_name] = (r, g, b)
        return self.colors[class_name]

def rotate_point_cw90(x, y, img_width, img_height):
    return img_height - 1 - y, x

def rotate_bbox_cw90(x1, y1, x2, y2, img_width, img_height):
    """
    Преобразует bbox (x1<=x2, y1<=y2) в повёрнутое изображение,
    используя все четыре угла.
    """
    corners = [(x1, y1), (x2, y1), (x1, y2), (x2, y2)]
    rotated = [rotate_point_cw90(cx, cy, img_width, img_height) for cx, cy in corners]
    xs = [p[0] for p in rotated]
    ys = [p[1] for p in rotated]
    return min(xs), min(ys), max(xs), max(ys)

def draw_annotations_on_image(image_path, pt_path, output_path=None, show=False, font_size=12, color_mapper=None):
    if color_mapper is None:
        color_mapper = ColorMapper()

    # Исходное изображение (до поворота)
    img_orig = Image.open(image_path).convert('RGB')
    W, H = img_orig.size   # W = ширина, H = высота

    # Загружаем PT-файл (формат: (cx, cy, w, h) относительные)
    data = torch.load(pt_path, map_location='cpu')
    nodes = data.get('x', [])   # список узлов

    # Поворачиваем изображение
    img_rot = img_orig.transpose(Image.ROTATE_270)  # 90° по часовой
    draw = ImageDraw.Draw(img_rot)

    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except:
        font = ImageFont.load_default()

    for node in nodes:
        cx_rel, cy_rel, w_rel, h_rel = node

        # Абсолютные координаты в исходном изображении
        x1 = (cx_rel - w_rel/2) * W
        y1 = (cy_rel - h_rel/2) * H
        x2 = (cx_rel + w_rel/2) * W
        y2 = (cy_rel + h_rel/2) * H

        # Преобразуем bbox в повёрнутую систему координат
        x1_rot, y1_rot, x2_rot, y2_rot = rotate_bbox_cw90(x1, y1, x2, y2, W, H)

        class_name = "test"   # укажите реальное имя, если есть
        color = color_mapper.get_color(class_name)

        # Рисуем прямоугольник
        draw.rectangle([x1_rot, y1_rot, x2_rot, y2_rot], outline=color, width=2)
        draw.text((x1_rot + 5, y1_rot + 5), class_name, fill=color, font=font)

        # Центр объекта
        cx_abs = cx_rel * W
        cy_abs = cy_rel * H
        cx_rot, cy_rot = rotate_point_cw90(cx_abs, cy_abs, W, H)
        r = 4
        draw.ellipse([cx_rot - r, cy_rot - r, cx_rot + r, cy_rot + r], fill='red', outline='red')

    if output_path:
        img_rot.save(output_path)
        print(f"Saved: {output_path}")
    if show:
        import matplotlib.pyplot as plt
        plt.imshow(np.array(img_rot))
        plt.axis('off')
        plt.title(os.path.basename(image_path))
        plt.show()

def visualize_all_annotations(image_root, pt_root, output_root=None, show=False, verbose=True, max_scenes=None):
    """
    max_scenes: если задано, обрабатывает только первые max_scenes сцен.
    """
    image_scenes_dir = os.path.join(image_root, 'scenes')
    if not os.path.isdir(image_scenes_dir):
        print(f"Папка scenes не найдена: {image_scenes_dir}")
        return

    scene_names = [d for d in os.listdir(image_scenes_dir) if os.path.isdir(os.path.join(image_scenes_dir, d))]
    #if max_scenes is not None:
    #    scene_names = scene_names[:max_scenes]
    if verbose:
        print(f"Найдено сцен: {len(scene_names)}")

    color_mapper = ColorMapper()

    for scene in scene_names:
        if scene != "0958220d-e2c2-2de1-9710-c37018da1883":
            continue
        scene_image_path = os.path.join(image_scenes_dir, scene, 'sequence')
        if not os.path.isdir(scene_image_path):
            continue

        scene_pt_path = os.path.join(pt_root, scene)
        if not os.path.isdir(scene_pt_path):
            if verbose:
                print(f"PT папка для сцены {scene} не найдена: {scene_pt_path}")
            continue

        image_pattern = os.path.join(scene_image_path, "**", "frame-*.color.jpg")
        image_paths = glob.glob(image_pattern, recursive=True)
        image_paths = [p for p in image_paths if '.rendered.' not in p]

        if verbose:
            print(f"Сцена {scene}: {len(image_paths)} изображений")

        for img_path in image_paths:
            base_name = os.path.basename(img_path)
            pt_name = base_name.replace('.color.jpg', '.pt')
            # Ищем .pt файл в scene_pt_path рекурсивно
            pt_pattern = os.path.join(scene_pt_path, "**", pt_name)
            candidates = glob.glob(pt_pattern, recursive=True)
            if not candidates:
                if verbose:
                    print(f"pt не найден: {pt_name} в {scene_pt_path}")
                continue
            pt_path = candidates[0]

            if output_root:
                rel_path = os.path.relpath(img_path, image_root)
                output_path = os.path.join(output_root, rel_path)
                os.makedirs(os.path.dirname(output_path), exist_ok=True)
                output_path = output_path.replace('.color.jpg', '.annotated.jpg')
            else:
                output_path = None

            draw_annotations_on_image(img_path, pt_path, output_path, show, color_mapper=color_mapper)

image_root = "/mnt/external_usb_hdd/6YL/Datasets/3RScan"
pt_root   = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/Splited_graphs"
output_root = "/home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated"
# Для теста обработаем только первые 2 сцены
visualize_all_annotations(image_root, pt_root, output_root, show=False, verbose=True, max_scenes=10)

Найдено сцен: 1481
Сцена 0958220d-e2c2-2de1-9710-c37018da1883: 192 изображений
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/0958220d-e2c2-2de1-9710-c37018da1883/sequence/frame-000040.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/0958220d-e2c2-2de1-9710-c37018da1883/sequence/frame-000077.annotated.jpg


/tmp/ipykernel_34501/2939056214.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(pt_path, map_location='cpu')


Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/0958220d-e2c2-2de1-9710-c37018da1883/sequence/frame-000030.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/0958220d-e2c2-2de1-9710-c37018da1883/sequence/frame-000116.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/0958220d-e2c2-2de1-9710-c37018da1883/sequence/frame-000001.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/0958220d-e2c2-2de1-9710-c37018da1883/sequence/frame-000087.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/0958220d-e2c2-2de1-9710-c37018da1883/sequence/frame-000092.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_graph_localization/data/graph_vis_rotated/scenes/0958220d-e2c2-2de1-9710-c37018da1883/sequence/frame-000127.annotated.jpg
Saved: /home/pinkin_ek/projects/Scene_gr

In [ ]:
pt_graph = torch.load("/mnt/external_usb_hdd/6YL/Datasets/3RScan/Splited_graphs/4a9a43e2-7736-2874-841a-f239715be4c6/frame-000000.pt")
print(pt_graph)

In [2]:
import math
import torch
import logging
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import multiprocessing
from os.path import join
from datetime import datetime
import new_src.dataset_gpt
import new_src.graph_model
import torchvision.transforms as transforms
from torch.utils.data.dataloader import DataLoader
torch.backends.cudnn.benchmark= True  # Provides a speedup

#import util
import argparse
import warnings
warnings.filterwarnings('ignore')
import os

DEFAULT_MEAN = [0.44420420130352495, 0.41322746532289134, 0.3678658064565412]
DEFAULT_STD = [0.24352604373543688, 0.24045797651069503, 0.24250136992133814]

logging.basicConfig(
    level=logging.DEBUG,  # <-- ключевое изменение
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
#### Initial setup: parser, logging...
import os
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image


def _load_img(path):
    if path is None or not os.path.exists(path):
        return None
    return Image.open(path).convert("RGB")


def _room_id(dataset, scene_name):
    return dataset.scan_to_ref.get(scene_name, scene_name)


def _get_graph_x_from_sample(sample):
    g = sample.get("graph", None)
    if g is None:
        return None

    # graph может быть dict / torch_geometric.Data / что-то похожее
    if isinstance(g, dict):
        x = g.get("x", None)
    else:
        x = getattr(g, "x", None)

    if x is None:
        return None

    if not torch.is_tensor(x):
        x = torch.as_tensor(x)

    if x.ndim == 1:
        x = x.view(1, -1)

    if x.shape[-1] < 2:
        return None

    return x.detach().cpu().float()


def _get_image_for_display(sample, mean=None, std=None):
    """
    Возвращает изображение в формате HWC numpy для matplotlib.

    Приоритет:
      1) sample["image"] если он уже есть
      2) sample["img_path"] если image нет
    """
    img = sample.get("image", None)

    if img is None:
        img_path = sample.get("img_path", None)
        if img_path is None:
            return None
        pil_img = _load_img(img_path)
        if pil_img is None:
            return None
        # fallback: как минимум покажем уже повернутую картинку
        pil_img = pil_img.transpose(Image.Transpose.ROTATE_270)  # 90° clockwise
        return pil_img

    # Если это PIL — просто вернём его
    if isinstance(img, Image.Image):
        return img

    # Если это tensor CHW, де-нормализуем и переводим в HWC
    if torch.is_tensor(img):
        x = img.detach().cpu()

        if x.ndim == 3 and x.shape[0] in (1, 3):
            x = x.float()

            # Если изображение похоже на нормализованное — де-нормализуем
            if mean is not None and std is not None:
                mean_t = torch.tensor(mean, dtype=x.dtype).view(-1, 1, 1)
                std_t = torch.tensor(std, dtype=x.dtype).view(-1, 1, 1)
                if mean_t.shape[0] == x.shape[0]:
                    x = x * std_t + mean_t

            x = x.clamp(0, 1)
            x = x.permute(1, 2, 0).numpy()
            return x

        # Уже HWC или другой формат
        return x.numpy()

    # Если это вдруг numpy / что-то ещё
    return img


def _show_sample(
    ax,
    sample,
    title="",
    max_boxes=None,
    coords_normalized=True,
    mean=None,
    std=None,
):
    """
    coords_normalized=True:
        x[:,0], x[:,1] считаются в [0,1] и умножаются на W/H.

    coords_normalized=False:
        x[:,0], x[:,1] считаются уже в пикселях.
    """
    img = _get_image_for_display(sample, mean=mean, std=std)

    if img is None:
        ax.set_title(title + "\n[no image]")
        ax.axis("off")
        return

    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

    # figure size from displayed image
    if isinstance(img, Image.Image):
        W, H = img.size
    else:
        # numpy HWC
        H, W = img.shape[:2]

    x = _get_graph_x_from_sample(sample)
    if x is None:
        return

    n = x.shape[0]
    if max_boxes is not None:
        n = min(n, max_boxes)

    for i in range(n):
        row = x[i].tolist()

        cx, cy = row[0], row[1]

        # Если есть width/height — рисуем bbox, иначе только центр
        bw = row[2] if len(row) > 2 else None
        bh = row[3] if len(row) > 3 else None

        if coords_normalized:
            cx_px = cx * W
            cy_px = cy * H
        else:
            cx_px = cx
            cy_px = cy

        # bbox
        if bw is not None and bh is not None:
            if coords_normalized:
                bw_px = bw * W
                bh_px = bh * H
            else:
                bw_px = bw
                bh_px = bh

            x1 = cx_px - bw_px / 2.0
            y1 = cy_px - bh_px / 2.0

            rect = patches.Rectangle(
                (x1, y1),
                max(1.0, bw_px),
                max(1.0, bh_px),
                linewidth=2,
                edgecolor="black",
                facecolor="none",
            )
            ax.add_patch(rect)

        ax.plot(cx_px, cy_px, marker="o", markersize=4, color="red")
        ax.text(cx_px + 3, cy_px + 3, str(i), color="red", fontsize=8)


def _get_sample_via_getitem(dataset, global_index):
    """
    Берёт item через __getitem__() в inference mode.
    """
    old_mode = getattr(dataset, "is_inference", None)
    if old_mode is not None:
        dataset.is_inference = True
    try:
        sample = dataset[global_index]
    finally:
        if old_mode is not None:
            dataset.is_inference = old_mode
    return sample


def visualize_triplet_images(
    dataset,
    triplets_global_indexes,
    num_triplets_to_show=1,
    max_boxes=30,
    coords_normalized=True,
    mean=None,
    std=None,
):
    """
    dataset: TripletsDataset
    triplets_global_indexes: Tensor[B, K]
        [query_index, positive_index, neg1, neg2, ...]
    coords_normalized:
        True  -> graph x/y в [0,1]
        False -> graph x/y уже в пикселях
    """
    if not torch.is_tensor(triplets_global_indexes):
        triplets_global_indexes = torch.as_tensor(triplets_global_indexes)

    B, K = triplets_global_indexes.shape
    num_triplets_to_show = min(num_triplets_to_show, B)

    for b in range(num_triplets_to_show):
        gidx = triplets_global_indexes[b].detach().cpu().tolist()

        q_local_idx = int(gidx[0])
        p_db_idx = int(gidx[1])
        neg_db_idx = [int(x) for x in gidx[2:]]

        # query в inference-mode лежит после database_items
        q_global_idx = dataset.database_num + q_local_idx

        q_sample = _get_sample_via_getitem(dataset, q_global_idx)
        p_sample = _get_sample_via_getitem(dataset, p_db_idx)
        n_samples = [_get_sample_via_getitem(dataset, i) for i in neg_db_idx]

        q_scene = q_sample.get("scene")
        p_scene = p_sample.get("scene")
        n_scenes = [s.get("scene") for s in n_samples]

        q_ref = _room_id(dataset, q_scene)
        p_ref = _room_id(dataset, p_scene)
        n_refs = [_room_id(dataset, s) for s in n_scenes]

        print("=" * 120)
        print(f"TRIPLET #{b}")
        print(
            f"QUERY    | q_local={q_local_idx} | q_global={q_global_idx} | "
            f"scene={q_scene} | ref={q_ref} | img={q_sample.get('img_path')}"
        )
        print(
            f"POSITIVE | db_idx={p_db_idx}     | scene={p_scene} | "
            f"ref={p_ref} | img={p_sample.get('img_path')}"
        )
        for i, s in enumerate(n_samples):
            print(
                f"NEGATIVE_{i+1} | db_idx={neg_db_idx[i]} | "
                f"scene={n_scenes[i]} | ref={n_refs[i]} | img={s.get('img_path')}"
            )

        n_plots = 2 + len(n_samples)
        fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 5))
        if n_plots == 1:
            axes = [axes]

        _show_sample(
            axes[0],
            q_sample,
            title=f"QUERY\nscene={q_scene}\nref={q_ref}\nidx={q_local_idx}",
            max_boxes=max_boxes,
            coords_normalized=coords_normalized,
            mean=mean,
            std=std,
        )
        _show_sample(
            axes[1],
            p_sample,
            title=f"POSITIVE\nscene={p_scene}\nref={p_ref}\nidx={p_db_idx}",
            max_boxes=max_boxes,
            coords_normalized=coords_normalized,
            mean=mean,
            std=std,
        )

        for j, s in enumerate(n_samples):
            _show_sample(
                axes[2 + j],
                s,
                title=f"NEGATIVE_{j+1}\nscene={n_scenes[j]}\nref={n_refs[j]}\nidx={neg_db_idx[j]}",
                max_boxes=max_boxes,
                coords_normalized=coords_normalized,
                mean=mean,
                std=std,
            )

        plt.tight_layout()
        plt.show()


def train_baseline(args):
    start_time = datetime.now()
    args.save_dir = join("logs", args.save_dir, start_time.strftime('%Y-%m-%d_%H-%M-%S'))
    logging.info(f"Arguments: {args}")
    logging.info(f"The outputs are being saved in {args.save_dir}")
    logging.info(f"Using {torch.cuda.device_count()} GPUs and {multiprocessing.cpu_count()} CPUs")


    #### Creation of Datasets
    logging.debug(f"Loading dataset {args.dataset_name} from folder {args.datasets_folder}")

    triplets_ds = dataset_gpt.TripletsDataset(args, args.datasets_folder, args.dataset_name, "train", args.negs_num_per_query)
    logging.info(f"Train query set: {triplets_ds}")

    # val_ds = BaseDataset(args, args.datasets_folder, args.dataset_name, "val")
    # logging.info(f"Val set: {val_ds}")

    test_ds = dataset_gpt.BaseDataset(args, args.datasets_folder, args.dataset_name, "test")
    logging.info(f"Test set: {test_ds}")

    #### Initialize model
    in_dim = 4
    model = graph_model.VPRGraphEncoder(in_dim=in_dim, hidden_dim=256, n_layers=3, num_node_classes=528 + 1, proj_dim=128).to(args.device)

    ### Setup Optimizer and Loss
    if args.optim == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
        # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=int(args.queries_per_epoch/args.train_batch_size), gamma=0.5, last_epoch=-1)
    elif args.optim == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=0.001)

    GlobalTriplet = nn.TripletMarginLoss(margin=args.margin, p=2, reduction="sum")

    #### Resume model, optimizer, and other training parameters
    if args.resume:
        #model, _, best_r5, start_epoch_num, not_improved_num = util.resume_train(args, model, strict=False)
        logging.info(f"Resuming from epoch {start_epoch_num} with best recall@5 {best_r5:.1f}")
    else:
        best_r5 = start_epoch_num = not_improved_num = 0

    #logging.info(f"Output dimension of the model is {args.features_dim}")

    #### Training loop
    for epoch_num in range(start_epoch_num, args.epochs_num):
        logging.info(f"Start training epoch: {epoch_num:02d}")
        
        epoch_start_time = datetime.now()
        epoch_losses = np.zeros((0,1), dtype=np.float32)
        
        # How many loops should an epoch last (default is 5000/1000=5)
        loops_num = math.ceil(args.queries_per_epoch / args.cache_refresh_rate)
        for loop_num in range(loops_num):
            logging.debug(f"Cache: {loop_num} / {loops_num}")
            
            # Compute triplets to use in the triplet loss
            triplets_ds.is_inference = True
            triplets_ds.compute_triplets(args, model)
            triplets_ds.is_inference = False
            
            triplets_dl = DataLoader(dataset=triplets_ds, num_workers=args.num_workers,
                                    batch_size=args.train_batch_size,
                                    shuffle=True,
                                    collate_fn=dataset_gpt.graph_collate_fn,
                                    pin_memory=(args.device=="cuda"),
                                    drop_last=True)
            
            model = model.train()
            print(len(triplets_ds), "Количество триплетов")
            # images shape: (train_batch_size*4)*3*H*W
            for batch_samples, _, triplets_global_indexes in tqdm(triplets_dl, ncols=100):
                # Compute features of all images (images contains queries, positives and negatives)
                
                if epoch_num == 0 and loop_num == 0:
                    visualize_triplet_images(
                        dataset=triplets_ds,
                        triplets_global_indexes=triplets_global_indexes,
                        num_triplets_to_show=20,
                        max_boxes=30,
                        coords_normalized=True,   # если graph['x'] в [0,1]
                        mean=DEFAULT_MEAN,
                        std=DEFAULT_STD,
                    )
                
                print(batch_samples)
                batch_graph = batch_samples["graph"].to(args.device)        
                global_features = model(batch_graph)
                embeddings = global_features
                total_loss = 0
                global_loss = 0

                if args.criterion == "triplet":       
                    K = 2 + args.negs_num_per_query
                    B = embeddings.shape[0] // K
                    embeddings = embeddings.view(B, K, -1)

                    queries = embeddings[:, 0]
                    positives = embeddings[:, 1]
                    negatives = embeddings[:, 2:]   # [B, N, D]

                    loss = 0.0
                    for j in range(negatives.shape[1]):
                        global_loss += GlobalTriplet(queries, positives, negatives[:, j])

                                    
                global_loss /= (args.train_batch_size * args.negs_num_per_query)
                            
                total_loss = global_loss


                del global_features

                optimizer.zero_grad()
                total_loss.backward()
                optimizer.step()

                batch_loss = total_loss.item()
                epoch_losses = np.append(epoch_losses, batch_loss)
                del total_loss

                print("total_loss", batch_loss)
            
            
            logging.debug(f"global loss = {global_loss.item():.6f}")
            logging.debug(f"Epoch[{epoch_num:02d}]({loop_num}/{loops_num}): " +
                        f"current batch triplet loss = {batch_loss:.4f}, " +
                        f"average epoch triplet loss = {epoch_losses.mean():.4f}")
            #recalls, recalls_str, _ = test(args, test_ds, model, device=args.device, ks=args.recall_values)
            #logging.info(f"Recalls on val set {test_ds}: {recalls_str}")

        
        logging.info(f"Finished epoch {epoch_num:02d} in {str(datetime.now() - epoch_start_time)[:-7]}, "
                    f"average epoch triplet loss = {epoch_losses.mean():.4f}")

        # Compute recalls on validation set
        """
        recalls, recalls_str = test.test(args, val_ds, model)
        logging.info(f"Reranking recalls on val set {val_ds}: {recalls_str}")
        
        is_best = recalls[1] > best_r5
        
        # Save checkpoint, which contains all training parameters
        util.save_checkpoint(args, {"epoch_num": epoch_num, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(), "recalls": recalls, "best_r5": best_r5,
            "not_improved_num": not_improved_num
        }, is_best, filename="last_model.pth")
        
        # If recall@5 did not improve for "many" epochs, stop training
        if is_best:
            logging.info(f"Improved: previous best R@5 = {best_r5:.1f}, current R@5 = {(recalls[1]):.1f}")
            best_r5 = recalls[1]
            not_improved_num = 0
        else:
            not_improved_num += 1
            logging.info(f"Not improved: {not_improved_num} / {args.patience}: best R@5 = {best_r5:.1f}, current R@5 = {(recalls[1]):.1f}")
            if not_improved_num >= args.patience:
                logging.info(f"Performance did not improve for {not_improved_num} epochs. Stop training.")
                break


    logging.info(f"Best R@5: {best_r5:.1f}")
    logging.info(f"Trained for {epoch_num+1:02d} epochs, in total in {str(datetime.now() - start_time)[:-7]}")

    #### Test best model on test set
    best_model_state_dict = torch.load(join(args.save_dir, "best_model.pth"))["model_state_dict"]
    model.load_state_dict(best_model_state_dict)

    recalls, recalls_str = test.test(args, test_ds, model, test_method=args.test_method)
    logging.info(f"Recalls on {test_ds}: {recalls_str}")
    """

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description='Training script for VPRGraphEncoder with triplet loss', allow_abbrev=False)
    
    parser.add_argument("--train_batch_size", type=int, default=1024,
                        help="Number of triplets (query, pos, negs) in a batch. Each triplet consists of 12 images")
    parser.add_argument("--infer_batch_size", type=int, default=1024,
                        help="Batch size for inference (caching and testing)")
    parser.add_argument("--criterion", type=str, default='triplet', help='loss to be used',
                        choices=["triplet", "sare_ind", "sare_joint"])
    parser.add_argument("--margin", type=float, default=0.1,
                        help="margin for the triplet loss")
    parser.add_argument("--epochs_num", type=int, default=50,
                        help="number of epochs to train for")
    parser.add_argument("--patience", type=int, default=3)
    parser.add_argument("--lr", type=float, default=0.00001, help="_")
    parser.add_argument("--optim", type=str, default="adam", help="_", choices=["adam", "sgd"])
    parser.add_argument("--cache_refresh_rate", type=int, default=1024,
                        help="How often to refresh cache, in number of queries")
    parser.add_argument("--queries_per_epoch", type=int, default=5000,
                        help="How many queries to consider for one epoch. Must be multiple of cache_refresh_rate")
    parser.add_argument("--negs_num_per_query", type=int, default=2,
                        help="How many negatives to consider per each query in the loss")
    parser.add_argument("--neg_samples_num", type=int, default=1000,
                        help="How many negatives to use to compute the hardest ones")
    parser.add_argument("--mining", type=str, default="partial", choices=["partial", "full", "random", "msls_weighted"])
    # Model parameters
    parser.add_argument("--l2", type=str, default="before_pool", choices=["before_pool", "after_pool", "none"],
                        help="When (and if) to apply the l2 norm with shallow aggregation layers")
    parser.add_argument('--pca_dim', type=int, default=None, help="PCA dimension (number of principal components). If None, PCA is not used.")
    parser.add_argument("--registers", action='store_true', help="_")
    parser.add_argument("--features_dim", type=int, default=128, help="_")

    # Initialization parameters
    parser.add_argument("--seed", type=int, default=0)
#    parser.add_argument("--foundation_model_path", type=str, default=None,
#                        help="Path to load foundation model checkpoint.")
    parser.add_argument("--resume", type=str, default=None,
                        help="Path to load checkpoint from, for resuming training or testing.")
    # Other parameters
    parser.add_argument("--device", type=str, default="cuda", choices=["cuda", "cpu"])
    parser.add_argument("--num_workers", type=int, default=8, help="num_workers for all dataloaders")
    parser.add_argument('--resize', type=int, default=[224,224], nargs=2, help="Resizing shape for images (HxW).")
    parser.add_argument('--dense_feature_map_size', type=int, default=[61,61,128], nargs=3, 
                        help="size of dense feature map (a 61x61 grid 128-dim local features)")
    parser.add_argument('--test_method', type=str, default="hard_resize",
                        choices=["hard_resize", "single_query", "central_crop", "five_crops", "nearest_crop", "maj_voting"],
                        help="This includes pre/post-processing methods and prediction refinement")
    parser.add_argument("--majority_weight", type=float, default=0.01, 
                        help="only for majority voting, scale factor, the higher it is the more importance is given to agreement")
    parser.add_argument("--efficient_ram_testing", action='store_true', help="_")
    parser.add_argument("--val_positive_dist_threshold", type=int, default=25, help="_")
    parser.add_argument("--train_positives_dist_threshold", type=int, default=10, help="_")
    parser.add_argument('--recall_values', type=int, default=[1,5,10,20], nargs="+",
                        help="Recalls to be computed, such as R@5.")
    parser.add_argument("--rerank_num", type=int, default=100, help="_")
    # Data augmentation parameters
    parser.add_argument("--modalities", nargs='+', choices=['image', 'graph', 'pose'], 
                    default=['graph', 'pose'], help="List of modalities")
    parser.add_argument("--brightness", type=float, default=None, help="_")
    parser.add_argument("--contrast", type=float, default=None, help="_")
    parser.add_argument("--saturation", type=float, default=None, help="_")
    parser.add_argument("--hue", type=float, default=None, help="_")
    parser.add_argument("--rand_perspective", type=float, default=None, help="_")
    parser.add_argument("--horizontal_flip", action='store_true', help="_")
    parser.add_argument("--random_resized_crop", type=float, default=None, help="_")
    parser.add_argument("--random_rotation", type=float, default=None, help="_")
    # Paths parameters
    parser.add_argument("--datasets_folder", type=str, default="/mnt/external_usb_hdd/6YL/Datasets", help="Path with all datasets")
    parser.add_argument("--dataset_name", type=str, default="3RScan", help="Relative path of the dataset")
    parser.add_argument("--pca_dataset_folder", type=str, default=None,
                        help="Path with images to be used to compute PCA (ie: pitts30k/images/train")
    parser.add_argument("--save_dir", type=str, default="/home/pinkin_ek/projects/Scene_graph_localization/data",
                        help="Folder name of the current run (saved in ./logs/)")
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignored unknown arguments: {unknown}")
        
    train_baseline(args)

ModuleNotFoundError: No module named 'dataset_gpt'